# Chapter 3: Calculus Foundations

Companion notebook for *The Math That Powers AI* (2nd ed.), Chapter 3.

Calculus is the engine of learning in AI: gradients tell us which direction to nudge
parameters, the chain rule lets us differentiate deep compositions (backpropagation),
and second derivatives (the Hessian) tell us about the shape of the loss surface.

All functions are imported from the `mathpowersai.calculus` package module.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np

from mathpowersai.calculus import (
    chain_rule_demo,
    classify_critical_point,
    numerical_gradient,
    numerical_hessian,
    sigmoid,
    sigmoid_derivative,
    vanishing_gradient_demo,
)

## 1. Numerical vs. analytic gradient

For $f(x, y) = x^2 + 2y^2$, the analytic gradient is
$\nabla f = [2x,\ 4y]$. At the chapter's test point $(3, 2)$ this gives $[6, 8]$.

`numerical_gradient` approximates the same thing with central finite differences,
$\partial f / \partial x_i \approx \big(f(x + \epsilon e_i) - f(x - \epsilon e_i)\big) / (2\epsilon)$,
and should match the analytic answer to high precision.

In [ ]:
f = lambda x: x[0]**2 + 2*x[1]**2
x = np.array([3.0, 2.0])

numerical = numerical_gradient(f, x)
analytical = np.array([2*x[0], 4*x[1]])

print(f"Numerical gradient:  {numerical}")   # [6, 8]
print(f"Analytical gradient: {analytical}")  # [6, 8]
print(f"Max abs difference:  {np.max(np.abs(numerical - analytical)):.2e}")
assert np.allclose(numerical, analytical, atol=1e-6)

## 2. The chain rule

Backpropagation is the chain rule applied at scale. The chapter's example composes
$y = e^u$ with $u = x^2$, so

$$\frac{dy}{dx} = \frac{dy}{du} \cdot \frac{du}{dx} = e^{x^2} \cdot 2x = 2x\,e^{x^2}.$$

`chain_rule_demo` returns each factor of the product, and we verify the result
against a finite-difference check.

In [ ]:
demo = chain_rule_demo(1.0)

print("Chain rule for y = e^(x^2) at x = 1:")
print(f"  dy/du = e^(x^2)    = {demo['dy_du']:.6f}")
print(f"  du/dx = 2x         = {demo['du_dx']:.6f}")
print(f"  dy/dx = 2x e^(x^2) = {demo['dy_dx']:.6f}")

numeric = numerical_gradient(lambda v: np.exp(v[0]**2), np.array([1.0]))[0]
print(f"  numerical check    = {numeric:.6f}")
assert abs(demo['dy_dx'] - numeric) < 1e-4

## 3. Sigmoid and its derivative

The sigmoid activation $\sigma(x) = 1 / (1 + e^{-x})$ has the elegant derivative
$\sigma'(x) = \sigma(x)\,(1 - \sigma(x))$.

Since $\sigma(0) = 0.5$, the derivative attains its **maximum value of 0.25 at $x = 0$**
and decays toward zero as $|x|$ grows — the seed of the vanishing-gradient problem.

In [ ]:
print("Sigmoid sigma(x) = 1 / (1 + e^(-x)):")
print(f"  sigma(0)  = {sigmoid(0.0):.4f}")
print(f"  sigma'(0) = sigma(0)(1 - sigma(0)) = {sigmoid_derivative(0.0):.4f} (maximum value)")
print(f"  sigma'(5) = {sigmoid_derivative(5.0):.6f}")

# The derivative never exceeds 0.25, anywhere on the real line.
xs = np.linspace(-10.0, 10.0, 2001)
print(f"  max of sigma'(x) over [-10, 10]: {sigmoid_derivative(xs).max():.4f}")
assert sigmoid_derivative(xs).max() <= 0.25 + 1e-12

## 4. Vanishing gradients: the $0.25^n$ decay

Backpropagating through $n$ sigmoid layers multiplies $n$ derivative factors
together. Each factor is at most $0.25$, so the activation contribution alone
shrinks like $0.25^n$ — after 10 layers it is already about $10^{-6}$.
This is why deep networks moved to activations like ReLU.

In [ ]:
factors = vanishing_gradient_demo(10)

print("Vanishing gradients through sigmoid layers:")
for k in (1, 2, 5, 10):
    print(f"  after {k:2d} layers: 0.25^{k} = {factors[k - 1]:.10f}")

assert np.isclose(factors[-1], 0.25**10)

## 5. Hessian-based critical-point classification

At a critical point (where $\nabla f = 0$), the eigenvalues of the Hessian
$H_{ij} = \partial^2 f / \partial x_i \partial x_j$ classify the local shape:

| Eigenvalues | Classification | Shape |
|---|---|---|
| all positive | local **minimum** | bowl |
| all negative | local **maximum** | inverted bowl |
| mixed signs | **saddle** point | horse saddle |

We compute the Hessian numerically with `numerical_hessian` at the origin for
three textbook surfaces and classify each with `classify_critical_point`.

In [ ]:
cases = [
    ("f(x, y) = x^2 + y^2", lambda v: v[0]**2 + v[1]**2),    # minimum
    ("f(x, y) = -x^2 - y^2", lambda v: -v[0]**2 - v[1]**2),  # maximum
    ("f(x, y) = x^2 - y^2", lambda v: v[0]**2 - v[1]**2),    # saddle
]
origin = np.array([0.0, 0.0])

print("Second derivative test at the critical point (0, 0):")
for name, func in cases:
    hess = numerical_hessian(func, origin)
    eigenvalues = np.linalg.eigvalsh(hess)
    label = classify_critical_point(hess, tol=1e-3)
    print(f"  {name}: eigenvalues [{eigenvalues[0]:.2f}, {eigenvalues[1]:.2f}] -> {label}")

labels = [classify_critical_point(numerical_hessian(func, origin), tol=1e-3)
          for _, func in cases]
assert labels == ["minimum", "maximum", "saddle"]

## Summary

- Finite differences let us **check analytic gradients** numerically ($[6, 8]$ at $(3, 2)$ for $f = x^2 + 2y^2$).
- The **chain rule** turns derivatives of compositions into products of factors — the heart of backpropagation.
- The sigmoid's derivative peaks at **0.25 at $x = 0$**, so stacking $n$ sigmoid layers shrinks gradients like $0.25^n$ — the **vanishing-gradient problem**.
- **Hessian eigenvalues** classify critical points as minima, maxima, or saddles — the multivariate second derivative test.